# Hexagon resolution — measured sensitivity & autocorrelation range

Turns the *modelled* resolution table into *measured* evidence from your real hunting grounds, and
estimates the spatial autocorrelation range (criterion 3) that the geometry-free analysis couldn't.

**Part A — resolution sweep.** For each candidate point-to-point size it builds the grid, assigns
every ground to one hexagon, and reports the measured grounds-per-occupied-cell distribution and the
**single-ground share** (the number you want low, so cells genuinely pool several grounds).

**Part B — variogram.** Fits a spherical variogram to ground-level harvest density for a chosen
species and reports the **range** — the distance beyond which grounds are no longer correlated —
which translates into a defensible cell size.

Grounds-per-cell is purely geometric, so one representative ground layer (e.g. 2017, the most
complete geometry) is enough; you don't need every year. `DEMO = True` runs the whole thing on
simulated grounds (with built-in spatial structure) so it works without the confidential data.

In [46]:
# ===== CONFIG =====
BORDER_PATH = r"Czech_shape/Czech_shp.shp"        # national border, for grid generation
GROUNDS_PATH = r"honitby_2017.shp"  # ONE representative joined ground layer
OUTDIR = r"resolution_analysis"

ID_COL   = "HONITBA"
NAME_COL = "NAZEVHONIT"                          # set None if absent
CANDIDATE_P2P_KM = [4.5, 6.2, 8.5, 9.6, 12.4]    # sizes to test
ASSIGN = "centroid"     # fast; grounds-per-cell is ~identical to largest-overlap (each ground -> 1 cell)

# Variogram: a metric column present on the ground layer; pick a continuously distributed species.
VARIOGRAM_METRIC = "Bag_RedD_3"
VARIO_MAXLAG_KM  = 60      # max separation to model
VARIO_NBINS      = 20
VARIO_SAMPLE     = 1500    # subsample of grounds for the pairwise computation

TARGET_CRS = 5514

DEMO        = False
DEMO_NGROUND = 4000

In [47]:
import os, math, warnings, numpy as np, pandas as pd, geopandas as gpd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely.ops import unary_union
from shapely.prepared import prep
from scipy.optimize import curve_fit
warnings.filterwarnings("ignore")
os.makedirs(OUTDIR, exist_ok=True)

border = gpd.read_file(BORDER_PATH).to_crs(TARGET_CRS)
LAND = unary_union(border.geometry.values); PL = prep(LAND)
MINX,MINY,MAXX,MAXY = LAND.bounds

def build_grid(p2p_m):
    R=p2p_m/2; dx,dy=math.sqrt(3)*R,1.5*R
    ang=[math.radians(60*k+90) for k in range(6)]
    polys=[]; row=0; y=MINY-2*R
    while y<=MAXY+2*R:
        x=MINX-2*R+((dx/2) if row%2 else 0)
        while x<=MAXX+2*R:
            polys.append(Polygon([(x+R*math.cos(a),y+R*math.sin(a)) for a in ang])); x+=dx
        y+=dy; row+=1
    g=gpd.GeoDataFrame(geometry=polys, crs=TARGET_CRS)
    g=g[g.geometry.apply(PL.intersects)].reset_index(drop=True)
    g["hex_id"]=np.arange(len(g))
    return g

## Input grounds (real or simulated)

In [48]:
_SP="WildBoar"
def _demo_grounds(n=DEMO_NGROUND, seed=1):
    from shapely.ops import voronoi_diagram
    from shapely.geometry import MultiPoint, Point
    rng=np.random.default_rng(seed)
    pts=[]
    while len(pts)<n:
        xs=rng.uniform(MINX,MAXX,n); ys=rng.uniform(MINY,MAXY,n)
        for px,py in zip(xs,ys):
            if PL.contains(Point(px,py)): pts.append((px,py))
            if len(pts)>=n: break
    cells=[c.intersection(LAND) for c in voronoi_diagram(MultiPoint(pts),envelope=LAND).geoms]
    rows=[{ID_COL:f"CZ{i:05d}", NAME_COL:("obora" if rng.random()<0.03 else f"g{i}"), "geometry":c}
          for i,c in enumerate(cells) if (not c.is_empty and c.area>0)]
    gdf=gpd.GeoDataFrame(rows,crs=TARGET_CRS)
    # density with a SMOOTH spatial trend (so the variogram has a real range) + noise
    cx=gdf.geometry.centroid.x.values; cy=gdf.geometry.centroid.y.values
    nc=40; cxs=rng.uniform(MINX,MAXX,nc); cys=rng.uniform(MINY,MAXY,nc)
    amp=rng.uniform(2,6,nc); sig=9000.0                       # patchy field, range ~ 2-3*sig
    field=np.zeros(len(gdf))
    for k in range(nc):
        field += amp[k]*np.exp(-(((cx-cxs[k])**2+(cy-cys[k])**2)/(2*sig**2)))
    dens=np.clip(4 + field + rng.normal(0,0.6,len(gdf)), 0, None)
    area_km2=gdf.geometry.area.values/1e6
    val=np.round(dens*area_km2).astype("float64")
    val[rng.random(len(gdf))<0.20]=np.nan
    gdf[VARIOGRAM_METRIC]=val
    return gdf

def read_grounds():
    if DEMO: return _demo_grounds()
    g=gpd.read_file(GROUNDS_PATH)
    if g.crs is None or g.crs.to_epsg()!=TARGET_CRS: g=g.to_crs(TARGET_CRS)
    return g

grounds = read_grounds()
print("Grounds loaded:", len(grounds))

Grounds loaded: 6245


In [49]:
grounds = read_grounds()
print(list(grounds.columns))

['HONITBA', 'NAZEVHONIT', 'year', 'honitba_1', 'Plan_RedDe', 'Plan_Red_1', 'Plan_Red_2', 'Plan_Red_3', 'Plan_Fallo', 'Plan_Fal_1', 'Plan_Fal_2', 'Plan_Fal_3', 'Plan_Moufl', 'Plan_Mou_1', 'Plan_Mou_2', 'Plan_Mou_3', 'Plan_RoeDe', 'Plan_Roe_1', 'Plan_Roe_2', 'Plan_Roe_3', 'Plan_WildB', 'Plan_Wil_1', 'Plan_Wil_2', 'Plan_Wil_3', 'Plan_Wil_4', 'Plan_SikaD', 'Plan_Sik_1', 'Plan_Sik_2', 'Plan_Sik_3', 'Plan_Sik_4', 'Plan_Sik_5', 'Plan_White', 'Plan_Whi_1', 'Plan_Whi_2', 'Plan_Whi_3', 'Plan_Chamo', 'Plan_Cha_1', 'Plan_Cha_2', 'Plan_Cha_3', 'Plan_Capra', 'Plan_Lepus', 'Plan_Oryct', 'Plan_Aythy', 'Plan_Ayt_1', 'Plan_Fulic', 'Plan_Phasi', 'Plan_Pha_1', 'Plan_Syrma', 'Plan_Syr_1', 'Plan_Numid', 'Plan_Num_1', 'Plan_Alect', 'Plan_AnasP', 'Plan_Anser', 'Plan_Ans_1', 'Plan_Ans_2', 'Bag_RedDee', 'Bag_RedD_1', 'Bag_RedD_2', 'Bag_RedD_3', 'Bag_Fallow', 'Bag_Fall_1', 'Bag_Fall_2', 'Bag_Fall_3', 'Bag_Mouflo', 'Bag_Mouf_1', 'Bag_Mouf_2', 'Bag_Mouf_3', 'Bag_RoeDee', 'Bag_RoeD_1', 'Bag_RoeD_2', 'Bag_RoeD_3', '

# Part A — measured resolution sweep

In [50]:
def assign_centroid(grounds, hexg):
    cen=grounds[[ID_COL]].copy(); cen["geometry"]=grounds.geometry.centroid
    cen=gpd.GeoDataFrame(cen,crs=grounds.crs)
    j=gpd.sjoin(cen, hexg[["hex_id","geometry"]], how="left", predicate="within")
    return j.dropna(subset=["hex_id"]).drop_duplicates(ID_COL)

rows=[]
per_cell={}
for p2p in CANDIDATE_P2P_KM:
    hexg=build_grid(p2p*1000)
    j=assign_centroid(grounds,hexg)
    counts=j.groupby("hex_id").size().values            # grounds per OCCUPIED cell
    per_cell[p2p]=counts
    rows.append(dict(
        p2p_km=p2p,
        cell_area_km2=round((3*math.sqrt(3)/2)*0.25*p2p**2,1),
        n_hex_total=len(hexg),
        n_hex_occupied=len(counts),
        pct_cells_occupied=round(100*len(counts)/len(hexg),1),
        mean_grounds=round(counts.mean(),2),
        median_grounds=int(np.median(counts)),
        p90_grounds=int(np.percentile(counts,90)),
        max_grounds=int(counts.max()),
        single_ground_pct=round(100*(counts==1).mean(),1),
    ))
sweep=pd.DataFrame(rows)
sweep.to_csv(os.path.join(OUTDIR,"resolution_sweep_measured.csv"),index=False)
sweep

,p2p_km,cell_area_km2,n_hex_total,n_hex_occupied,pct_cells_occupied,mean_grounds,median_grounds,p90_grounds,max_grounds,single_ground_pct
0,4.5,13.2,6301,4721,74.9,1.32,1,2,21,73.0
1,6.2,25.0,3375,3043,90.2,2.05,2,3,20,31.3
2,8.5,46.9,1829,1717,93.9,3.64,4,5,20,7.3
3,9.6,59.9,1456,1364,93.7,4.58,4,7,27,4.7
4,12.4,99.9,893,845,94.6,7.39,7,11,40,3.9


In [51]:
fig,ax1=plt.subplots(figsize=(8,4.8))
ax1.plot(sweep.p2p_km, sweep.single_ground_pct, "o-", color="#c0392b", lw=2,
         label="% occupied cells with a single ground")
ax1.set_xlabel("hexagon point-to-point size (km)")
ax1.set_ylabel("single-ground share of occupied cells (%)", color="#c0392b")
ax1.axhline(50,color="grey",ls=":",lw=0.8)
ax2=ax1.twinx()
ax2.plot(sweep.p2p_km, sweep.mean_grounds, "s--", color="#1769aa", lw=2,
         label="mean grounds / occupied cell")
ax2.set_ylabel("mean grounds per occupied cell", color="#1769aa")
ax1.set_title("Measured aggregation gain vs hexagon size\n(lower red = cells truly pool several grounds)")
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR,"single_ground_share.png"),dpi=130)
print("saved single_ground_share.png")

saved single_ground_share.png


In [52]:
print([c for c in grounds.columns if c.startswith("Bag")])

['Bag_RedDee', 'Bag_RedD_1', 'Bag_RedD_2', 'Bag_RedD_3', 'Bag_Fallow', 'Bag_Fall_1', 'Bag_Fall_2', 'Bag_Fall_3', 'Bag_Mouflo', 'Bag_Mouf_1', 'Bag_Mouf_2', 'Bag_Mouf_3', 'Bag_RoeDee', 'Bag_RoeD_1', 'Bag_RoeD_2', 'Bag_RoeD_3', 'Bag_WildBo', 'Bag_Wild_1', 'Bag_Wild_2', 'Bag_Wild_3', 'Bag_Wild_4', 'Bag_SikaDe', 'Bag_Sika_1', 'Bag_Sika_2', 'Bag_Sika_3', 'Bag_Sika_4', 'Bag_Sika_5', 'Bag_Sika_6', 'Bag_Sika_7', 'Bag_WhiteT', 'Bag_Whit_1', 'Bag_Whit_2', 'Bag_Whit_3', 'Bag_Chamoi', 'Bag_Cham_1', 'Bag_Cham_2', 'Bag_Cham_3', 'Bag_CapraA', 'Bag_LepusE', 'Bag_Orycto', 'Bag_Aythya', 'Bag_Ayth_1', 'Bag_Fulica', 'Bag_Phasia', 'Bag_Phas_1', 'Bag_Syrmat', 'Bag_Syrm_1', 'Bag_Numida', 'Bag_Numi_1', 'Bag_Alecto', 'Bag_AnasPl', 'Bag_AnserA', 'Bag_AnserF', 'Bag_Anse_1']


# Part B — autocorrelation range (variogram)

Spherical model fitted to ground-level density (`metric / ground area`) for `VARIOGRAM_METRIC`.
The **range** is where the curve plateaus; beyond it grounds are effectively uncorrelated.

In [53]:
def empirical_variogram(x, y, z, maxlag, nbins):
    P=np.c_[x,y]
    dx=P[:,0][:,None]-P[:,0][None,:]
    dy=P[:,1][:,None]-P[:,1][None,:]
    h=np.sqrt(dx*dx+dy*dy)
    g=0.5*(z[:,None]-z[None,:])**2
    iu=np.triu_indices(len(z),k=1)
    h=h[iu]; g=g[iu]
    m=h<=maxlag; h=h[m]; g=g[m]
    edges=np.linspace(0,maxlag,nbins+1); cen=0.5*(edges[:-1]+edges[1:])
    idx=np.digitize(h,edges)-1
    gamma=np.array([g[idx==b].mean() if np.any(idx==b) else np.nan for b in range(nbins)])
    npairs=np.array([np.sum(idx==b) for b in range(nbins)])
    return cen, gamma, npairs

def spherical(h, nugget, sill, rng):
    h=np.asarray(h,float)
    out=np.where(h<rng, nugget+sill*(1.5*h/rng-0.5*(h/rng)**3), nugget+sill)
    return out

g=grounds.dropna(subset=[VARIOGRAM_METRIC]).copy()
g["dens"]=g[VARIOGRAM_METRIC].values/(g.geometry.area.values/1e6)
g=g[np.isfinite(g["dens"])]
rng_=np.random.default_rng(0)
take=rng_.choice(len(g), size=min(VARIO_SAMPLE,len(g)), replace=False)
gs=g.iloc[take]
x=gs.geometry.centroid.x.values/1000.0; y=gs.geometry.centroid.y.values/1000.0
z=gs["dens"].values

cen,gamma,npairs=empirical_variogram(x,y,z,VARIO_MAXLAG_KM,VARIO_NBINS)
ok=np.isfinite(gamma)
p0=[np.nanmin(gamma), np.nanmax(gamma)-np.nanmin(gamma), VARIO_MAXLAG_KM/2]
popt,_=curve_fit(spherical, cen[ok], gamma[ok], p0=p0,
                 bounds=([0,0,1],[np.inf,np.inf,VARIO_MAXLAG_KM]), maxfev=10000)
nugget,sill,vrange=popt
print(f"Variogram fit ({VARIOGRAM_METRIC}): range = {vrange:.1f} km  (nugget={nugget:.2f}, sill={sill:.2f})")
print(f"Suggested cell point-to-point to resolve structure: ~{vrange/2:.1f}-{vrange/3:.1f} km (<= range/2)")

Variogram fit (Bag_RedD_3): range = 1.1 km  (nugget=6.14, sill=0.00)
Suggested cell point-to-point to resolve structure: ~0.6-0.4 km (<= range/2)


In [54]:
fig,ax=plt.subplots(figsize=(8,4.8))
ax.scatter(cen, gamma, s=np.clip(npairs/npairs.max()*120,10,120), color="#1769aa", label="empirical")
hh=np.linspace(0,VARIO_MAXLAG_KM,200)
ax.plot(hh, spherical(hh,*popt), color="#c0392b", lw=2, label="spherical fit")
ax.axvline(vrange, color="grey", ls="--", lw=1); ax.text(vrange,ax.get_ylim()[0],f" range={vrange:.0f} km",color="grey",va="bottom")
ax.set_xlabel("separation distance (km)"); ax.set_ylabel("semivariance")
ax.set_title(f"Variogram of ground-level density — {VARIOGRAM_METRIC}")
ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUTDIR,"variogram.png"),dpi=130)
print("saved variogram.png")

saved variogram.png


# Outputs & how to read them

`resolution_analysis/` holds `resolution_sweep_measured.csv`, `single_ground_share.png`, and
`variogram.png`.

**Choosing the resolution, defensibly:**
- The **single-ground share** (Part A) is your primary evidence: pick the coarsest size where it has
  dropped enough that most cells pool several grounds (i.e. the aggregation is doing real work) — the
  curve's knee.
- Cross-check with the **variogram range** (Part B): a cell around range/2 resolves the spatial
  structure without redundant oversampling. If Part A's knee and range/2 agree, that convergence is
  the strongest justification you can report.
- For the paper, report this **measured** sweep table and the variogram instead of any modelled
  occupancy figures.

To run for real: set `DEMO = False`, point `GROUNDS_PATH` at one joined ground layer (2017), set
`VARIOGRAM_METRIC` to a real column (a continuously distributed species such as wild boar or red
deer reads best), and run top to bottom. Re-run Part B for a second species to check the range is
robust across species.